In [10]:
import pandas as pd

In [11]:
# working on python3.6
from fastai.text import *
path = Path('/data/asa/id_data')

SyntaxError: invalid syntax (text.py, line 8)

In [ ]:
# load data for LM training
# lmtrain_20180501-20180531_all.csv
# lmtrain_20180501-20180531_http.csv
filename = 'lmtrain_20180501-20180531_all.csv'
filename = 'lmtrain_20180501-20180531_http.csv'
data_lm = TextLMDataBunch.from_csv(path, filename, text_cols='RAW_PACKET')

In [ ]:
features = ['waitfor_delay', 'benchmark', 'sleep', 'and_or', 'select_statement', 
            'conditional', 'union_select', 'union_select_column_count', 'union_select_column_content',
            'char_encoding', 'db_schema', 'xp_cmdshell', 'db_procedure', 'db_declare', 'db_manipulation']

In [8]:
# load data for flassifier
classifier_filename = '/data/asa/id_data/SQL Injection.xlsx'
df = pd.read_excel(classifier_filename, encoding='cp949')
df = df.sample(frac=1.0)

# rename columns
df = df.rename(columns={
    'schema': 'db_schema',
    'procedure': 'db_procedure',
    'declare': 'db_declare',
})

ImportError: Install xlrd >= 1.0.0 for Excel support

In [9]:
for f in features:
    df.loc[:, f] = [f if v == 1 else '' for v in df[f].values.tolist()]

NameError: name 'features' is not defined

In [32]:
df.loc[:,'label'] = \
    df['waitfor_delay'].map(str) + ',' \
        + df['benchmark'].map(str) + ',' \
        + df['sleep'].map(str) + ',' \
        + df['and_or'].map(str) + ',' \
        + df['select_statement'].map(str) + ',' \
        + df['conditional'].map(str) + ',' \
        + df['union_select'].map(str) + ',' \
        + df['union_select_column_count'].map(str) + ',' \
        + df['union_select_column_content'].map(str) + ',' \
        + df['char_encoding'].map(str) + ',' \
        + df['db_schema'].map(str) + ',' \
        + df['xp_cmdshell'].map(str) + ',' \
        + df['db_procedure'].map(str) + ',' \
        + df['db_declare'].map(str) + ',' \
        + df['db_manipulation'].map(str)

NameError: name 'df' is not defined

In [46]:
df[['RAW_PACKET','label']+features].head(5)

,RAW_PACKET,label,waitfor_delay,benchmark,sleep,and_or,select_statement,conditional,union_select,union_select_column_count,union_select_column_content,char_encoding,db_schema,xp_cmdshell,db_procedure,db_declare,db_manipulation
2616,Protocol Name=TCP\\n target-ip-addr-start=10.0...,",,,,select_statement,,union_select,,,,,,,,",,,,,select_statement,,union_select,,,,,,,,
18784,Protocol Name=TCP\\n target-ip-addr-start=61.2...,",,,and_or,select_statement,,,,,char_encoding,d...",,,,and_or,select_statement,,,,,char_encoding,db_schema,,,,
17139,Protocol Name=TCP\\n target-ip-addr-start=211....,",,,,,,union_select,union_select_column_count,,...",,,,,,,union_select,union_select_column_count,,,,,,,
13198,Protocol Name=TCP\\n target-ip-addr-start=61.1...,"waitfor_delay,,,,,,,,,,,,,,",waitfor_delay,,,,,,,,,,,,,,
2063,Protocol Name=TCP\\n target-ip-addr-start=117....,",,,,,,,,,,,,,,",,,,,,,,,,,,,,,


In [47]:
# split dataset
n_train = int(df.shape[0]*0.8)
train_df = df[:n_train]
valid_df = df[n_train:]

train_df.shape, valid_df.shape

((19202, 25), (4801, 25))

In [48]:
bs = 128
data_clas = TextClasDataBunch.from_df(path, train_df=train_df, valid_df=valid_df, 
                                  vocab=data_lm.vocab, 
                                  text_cols='RAW_PACKET', 
                                  label_cols='label',
                                  label_delim=',',
                                  bs=bs)

In [49]:
gen = iter(data_clas.train_dl)
x, y = next(gen)
x.shape, y.shape

(torch.Size([128, 845]), torch.Size([128, 12]))

In [51]:
y

tensor([[1., 0., 0.,  ..., 0., 1., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')

In [52]:
learn_scratch = language_model_learner(data_lm, AWD_LSTM, drop_mult=0.5, pretrained=False)
learn_scratch.fit_one_cycle(3, 1e-2)

In [15]:
learn_scratch.save('model-{}-{}'.format(filename, 20190613))

In [56]:
learn_scratch.save('model_encoder-{}-{}'.format(filename, 20190613))

In [18]:
pickle_out = open('dict.pkl', 'wb')
pickle.dump(learn_scratch.vocab.itos, pickle_out)
pickle_out.close()

AttributeError: 'LanguageLearner' object has no attribute 'vocab'

In [55]:
def precision(log_preds, targs, thresh=0.5, epsilon=1e-8):
    pred_pos = (log_preds > thresh).float()
    tpos = torch.mul((targs == pred_pos).float(), targs.float())
    return (tpos.sum()/(pred_pos.sum() + epsilon))#.item()

def recall(log_preds, targs, thresh=0.5, epsilon=1e-8):
    pred_pos = (log_preds > thresh).float()
    tpos = torch.mul((targs == pred_pos).float(), targs.float())
    return (tpos.sum()/(targs.sum() + epsilon))

In [64]:
learn = text_classifier_learner(data_clas, AWD_LSTM, drop_mult=0.5)
learn.load_encoder('model_encoder-{}-{}'.format(filename, 20190613))
#learn.load('model-{}-{}'.format(filename, 20190613))



#learn.metrics = [accuracy_thresh, precision, recall]
#learn.load_encoder('model_encoder-{}-{}'.format(filename, 20190613))

RuntimeError: Error(s) in loading state_dict for AWD_LSTM:
	Missing key(s) in state_dict: "encoder.weight", "encoder_dp.emb.weight", "rnns.0.weight_hh_l0_raw", "rnns.0.module.weight_ih_l0", "rnns.0.module.weight_hh_l0", "rnns.0.module.bias_ih_l0", "rnns.0.module.bias_hh_l0", "rnns.1.weight_hh_l0_raw", "rnns.1.module.weight_ih_l0", "rnns.1.module.weight_hh_l0", "rnns.1.module.bias_ih_l0", "rnns.1.module.bias_hh_l0", "rnns.2.weight_hh_l0_raw", "rnns.2.module.weight_ih_l0", "rnns.2.module.weight_hh_l0", "rnns.2.module.bias_ih_l0", "rnns.2.module.bias_hh_l0". 
	Unexpected key(s) in state_dict: "model", "opt". 

In [20]:
import os
os.listdir('/data/asa/id_data')

['Path Traversal.xlsx',
 'Remote Code Inclusion.xlsx',
 'Remote Command Injection.xlsx',
 'Server Side Script.xlsx',
 'SQL Injection.xlsx',
 'XSS.xlsx',
 'lmtrain_20180501-20180531_all.csv',
 'lmtrain_20180501-20180531_http.csv',
 'lmtrain_20180501-20180531_all.csv_',
 'models',
 'train.csv',
 'train.xlsx',
 'string_pattern.txt',
 'sql_injection_labeled_final.xlsx',
 '_train_cp949.csv',
 '_train_latin1.csv']

In [ ]:
learn_scratch.predict("Protocol", 100)